# Run Match

In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas, MusicDBPermDir
from musicdb import PanDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import MatchDBDataIO, AlbumReq, NameReq, MatchReq, MatchDB, CrossMatchDB, PanDBMatch
io = FileIO()

In [ ]:
baseReqs = {"MetalArchives": 4,
            "RateYourMusic": 20,
            "Beatport": 35,
            "Spotify": 20,
            "Discogs": 3,  ## 12
            "Traxsource": 100000}
#baseDB    = "Beatport"
#baseDB    = "Discogs"
#baseDB    = "Spotify"
#baseDB    = "Traxsource"
#baseDB    = "MyMixTapez"
#baseDB    = "Genius"
#baseDB    = "MetalArchives"
baseDB    = "RateYourMusic"  # 3

minL = 1
maxL = 6

minA = 1
maxA = 30000000

#baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1))}
baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=minA, max=maxA))}
#baseReq   = {baseDB: AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1)}
#baseReq   = {baseDB: AlbumReq(min=10, max=baseReqs.get(baseDB,10000)+1, rnd=10000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "Beatport"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz", "Beatport", "Traxsource", "Genius", "MyMixTapez", "MetalArchives"] # "LastFM", "Deezer"]
#compareDBs  = ["RateYourMusic", "Traxsource", "Spotify", "Beatport"]
compareReqs = {compareDB: MatchReq(NameReq(min=minL-5, max=maxL+5), AlbumReq(min=3)) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

matchReqs  = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
mediaTypes = list(MasterMetas().getMedias().values())
reqs       = {"Media": mediaTypes, "Reqs": matchReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Primary Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

crossreqs  = {"Media": mediaTypes, "Reqs": {baseDB: MatchReq(AlbumReq(min=2))}, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Cross Match Run Params:")
print("  ==> DBs:   {0} x [{1}]".format(list(compareReqs.keys()), baseDB))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(crossreqs["Match"]))

In [ ]:
baseIO = MatchDBDataIO(db=baseDB, mediaTypes=reqs["Media"], mask=reqs["Mask"], verbose=True, base=True)
baseIO.loadNames()
baseIO.setAvailableNames(reqs["Reqs"][baseDB])
del baseIO

# Match & Cross Match

In [ ]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()
mdb.save()
del mdb

In [ ]:
mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
cmdb  = CrossMatchDB(baseDB, mres, crossreqs, verbose=True)
cmdb.match()
cmdb.save()

del cmdb

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=5)
pdbm.master()
pdbm.merge(allownew=True, verbose=False)

In [ ]:
pdbm.mergeMultiRows()

# Extra Match

In [ ]:
# 
# 2f60ed24b5   | Spotify        5NokRbYYfqacBmRVBRj0wD                    4   8    | THE VIOLENTS                            THE VIOLENT
# d747043fb9   | Discogs        415652                                    3   5    | TOM SMOTHERS                            TOMMY SMOTHERS
# 8b1aa6c9a2   | Beatport       689029                                    4   5    | THE MARÍAS                             THE MARIAS
# e5bad6da5e   | Discogs        282127                                    3   5    | TIM WILSON                              KIM WILSON
# d5d88bed85   | Discogs        1175923                                   4   5    | RIO GRANDE                              TRIO GRANDE
# a0f7076df1   | MetalArchives  41179                                     3   7    | SERPENTS                                SERPENT


In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=3, maxName=5)

In [ ]:
pdbm.include("""
824e7dfaad   | Spotify        4eukYJoDozkbKLrImwYWDZ                    3   8    | TEAM 600                                TEAM600
2079135a99   | Discogs        3142698                                   3   8    | CAM'-FLAGE                              CAM-FLAGE
a1ed0e4433   | Discogs        751906                                    4   8    | LIL BLACKY                              LIL' BLACKY
248f1507d9   | Genius         2715457                                   3   5    | JEWELXXET                               JEWEL XXET
9ad35f353a   | Genius         13180                                     3   8    | LIL' DUVAL                              LIL DUVAL
b41beaf653   | MyMixTapez     51114                                     3   5    | LIL' DUVAL                              LIL DUVAL
03a9d266cc   | Spotify        6YiHht3u7FFszle72kpbdQ                    3   8    | LIL' DUVAL                              LIL DUVAL
a20895c776   | MusicBrainz    119475576907088648053765796634321486618   4   5    | TITO GOMEZ                              TITO GÓMEZ
20427f2210   | Spotify        1VESEn29cZFpmsWMrpHyQT                    3   6    | POLAR BEAR                              POLARBEAR
630dccbcc1   | MusicBrainz    183780671351234893652194596767338612639   4   8    | SINSEMILIA                              SINSÉMILIA
1555f84b5d   | Spotify        7xtiaP8V8z95pASVuIrCRY                    4   8    | SINSEMILIA                              SINSÉMILIA
8b930cb38e   | Discogs        256048                                    3   7    | POLAR BEAR                              POLARBEAR
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5, minName=4)

In [ ]:
pdbm.include("""
54e6f6f8f0   | Discogs        51773                                     5   4    | BEN CENAC                               BEN CENAC
51d0e3f44b   | Spotify        5uhbWJFWcvQp3MSZicyoSs                    5   4    | MARC DEX                                MARC DEX
26f4b45f2c   | Genius         7267                                      5   4    | MAGIC JUAN                              MAGIC JUAN
1d6678588f   | MetalArchives  24681                                     5   4    | CHAINSAW                                CHAINSAW
88e8c427c6   | Genius         97765                                     5   4    | TIM WILSON                              TIM WILSON
c5b8ea857c   | Genius         365706                                    5   4    | INSTITUTE                               INSTITUTE
86f1b58e52   | Beatport       1161                                      5   4    | DJ MICRO                                DJ MICRO
2a1f0f3caa   | Genius         1567809                                   5   4    | OTIS GIBBS                              OTIS GIBBS
a9b6a06488   | Genius         357072                                    5   4    | THE EVENS                               THE EVENS
dd76dde9e5   | Genius         346193                                    5   4    | MIKE SCOTT                              MIKE SCOTT
0d9f5f1f8a   | Discogs        292436                                    5   4    | THE ACES                                THE ACES
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4, minName=4)

In [ ]:
pdbm.include("""
0b564bcb35   | Genius         374807                                    5   2    | NIK TURNER                              NIK TURNER
6573cdd1ec   | Beatport       80949                                     5   2    | MR. BIZZ                                MR. BIZZ
ac8734e175   | Discogs        3263089                                   5   3    | MARY HART                               MARY HART
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2, minName=5)

In [ ]:
pdbm.include("""
9099c55638   | Genius         386544                                    5   1    | LOS BURROS                              LOS BURROS
5a1c002961   | Genius         1153856                                   5   1    | JOHN FIELD                              JOHN FIELD
fb97692299   | Discogs        200586                                    5   1    | SHA ROCK                                SHA ROCK
059bd8f331   | Discogs        810250                                    5   1    | TITO GOMEZ                              TITO GOMEZ
43c9750700   | Spotify        7kZDKqfAuL3o74dJ1zCZG6                    5   1    | DJ MICRO                                DJ MICRO
dcc121a1c1   | Traxsource     89200                                     5   1    | DJ MICRO                                DJ MICRO
079e5e0497   | Genius         2303150                                   5   1    | PETR MUK                                PETR MUK
e5a94f053f   | Genius         381687                                    5   1    | JOE CUBA                                JOE CUBA
298dea30e7   | Discogs        753829                                    5   1    | ZWEISTEIN                               ZWEISTEIN
b18b859fd8   | Spotify        3lQGzIxapa9w0x84LIAKPI                    5   1    | THE ACES                                THE ACES
""")

pdbm.master()
pdbm.merge()

In [ ]:
del pdbm

In [ ]:
# Fix This:
#afc404588f   | 183166                   Discogs        113700                                  2    STEVE MASTERSON                                   STEVE MAESTRO                                      | afc404588f
#a6da8c9ee9   | 25875                    Discogs        678236                                  4    ALTAR OF FLESH                                    ALTAR OF FLIES                                     | a6da8c9ee9

# New Matching Code

# New Single Matching Code

In [ ]:
names = smdb.baseIO.getAvailableNames()

In [ ]:
smdb = SingleMatchDB(baseDB="RateYourMusic", compareDB="Spotify", reqs=reqs)
smdb.match()
smdb.save()
del smdb


mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
scmdb = SingleCrossMatchDB(baseDB, mres, crossreqs, verbose=True)
scmdb.match()
scmdb.save()
del scmdb


In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()

In [ ]:
mmeDF[mmeDF["RateYourMusic"] == '106836']

In [ ]:
mmeDF[mmeDF["Spotify"] == '3lk3F4u5qqxq8zFjwNj5U1']

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.masterSingle()
#pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
1d2402a17a   | 1061693                  Spotify        49D8h67pxvvUNGOLKEGjkx                  4    OWAIN ARWEL HUGHES                                OWAIN ARWEL HUGHES                                 | 1d2402a17a
a86b2ef789   | 121809                   Spotify        6NSIW1uEq8JZmxEkHMF17c                  4    ANNA TOMOWA-SINTOW                                ANNA TOMOWA-SINTOW                                 | a86b2ef789
877d262f5e   | 142182                   Spotify        5DwQvVHPVspRvStEAN722N                  4    TAKÁCS QUARTET                                   TAKÁCS QUARTET                                    | 877d262f5e
a2f65f8447   | 412578                   Spotify        50skve7Y0al39yGqLuCMNu                  4    MAURICE ABRAVANEL                                 MAURICE ABRAVANEL                                  | a2f65f8447
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
9554709d72   | 405351                   Spotify        2mHCS8PPaV7cAmT3ew8qHY                  2    SAULIUS SONDECKIS                                 SAULIUS SONDECKIS                                  | 9554709d72
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
3178f6847b   | 337551                   Spotify        7N0fh2csz0eFkrE01LF1m3                  1    STRATOS PAGIOUMTZIS                               STRATOS PAGIOUMTZIS                                | 3178f6847b
842d333cee   | 375588                   Spotify        2LqWWIvCBaetjLStxk1XK6                  1    VAN AND SCHENCK                                   VAN & SCHENCK                                      | 842d333cee
60cc9bc61a   | 77193                    Spotify        6VeTIJi6Dlx8ywPfIwqALY                  1    ALBERT NICHOLAS                                   ALBERT NICHOLAS                                    | 60cc9bc61a
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)